<a href="https://colab.research.google.com/github/GilangAngga12/Pemrosesan-teks-praktik/blob/main/timnas15okt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing Teks

In [3]:
 %pip install polars nltk Sastrawi

## Library

In [4]:
import re
import html
import polars as pl
import nltk
# import spacy
import unicodedata
from functools import lru_cache
from collections import Counter

# import pandas as pd

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [5]:
nltk.download("stopwords")

# Model spaCy untuk bahasa Indonesia
# spacy.cli.download("xx_ent_wiki_sm")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Helper Fungsion

In [6]:
def to_list(v):
    """map_elements memberi pl.Series untuk kolom list -> ubah ke list Python."""
    if v is None: return []
    if isinstance(v, pl.Series): return v.to_list()
    return list(v)

def _apply_str(df, col, out, fn):
    return df.with_columns(pl.col(col).map_elements(fn, return_dtype=pl.Utf8).alias(out))

def _apply_list(df, col, out, fn):
    return df.with_columns(
        pl.col(col).map_elements(lambda v: fn(to_list(v)), return_dtype=pl.List(pl.Utf8)).alias(out))

## Dataset

In [49]:
df = pl.read_csv("/content/dataset_twitter-x-data-tweet-scraper-pay-per-result-cheapest_timnas_2026-10-08_10-41-29-564.csv")

In [8]:
df

type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,lang,bookmarkCount,isReply,inReplyToId,conversationId,inReplyToUserId,inReplyToUsername,isPinned,author,extendedEntities,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,isQuote
str,i64,str,str,str,str,i64,i64,i64,i64,i64,str,str,i64,bool,i64,i64,i64,str,bool,str,str,str,str,str,str,str,str,str,bool,i64,bool
"""tweet""",2108447840284668323,"""https://x.com/Jezaelans/status…","""https://twitter.com/Jezaelans/…","""@MenteriPekerja Mknya dr awal …",null,0,0,0,0,1,"""Fri Oct 09 06:41:52 +0000 2026""","""in""",0,true,2108337734675017861,2108337734675017861,1717188745227862016,"""MenteriPekerja""",false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'user_mentions': [{'id_str': …","""{'rest_id': '17171887452278620…",null,null,null,false,0,null
"""tweet""",2108446988073402729,"""https://x.com/rang_simabua/sta…","""https://twitter.com/rang_simab…","""Pernyataan Rocky Gerung yang m…",null,2,0,2,0,30,"""Fri Oct 09 06:38:29 +0000 2026""","""in""",0,false,null,2108446988073402729,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'hashtags': [], 'symbols': []…",null,"""{'rest_id': '21081447174480077…","""{'type': 'tweet', 'id': '21081…",null,false,0,true
"""tweet""",2108446836512141677,"""https://x.com/BiLLRaY2019/stat…","""https://twitter.com/BiLLRaY201…","""Wantimpres angkat bicara..! R…",null,0,0,0,0,46,"""Fri Oct 09 06:37:53 +0000 2026""","""in""",0,false,null,2108446836512141677,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{}""",null,null,null,null,false,0,null
"""tweet""",2108445787856458122,"""https://x.com/triwahyuutama15/…","""https://twitter.com/triwahyuut…","""Pembentukan Wantimpres oleh Pr…",null,0,0,0,0,5,"""Fri Oct 09 06:33:43 +0000 2026""","""in""",0,false,null,2108445787856458122,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,null,null,null,false,0,null
"""tweet""",2108445386075721812,"""https://x.com/Putra_Riau11/sta…","""https://twitter.com/Putra_Riau…","""Ternyata 1 orang seperti anda …",null,0,0,0,0,4,"""Fri Oct 09 06:32:07 +0000 2026""","""in""",0,false,null,2108445386075721812,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{}""",null,null,null,null,false,0,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tweet""",2108179303100002598,"""https://x.com/kodam_siliwangi/…","""https://twitter.com/kodam_sili…","""Kasdam III/Siliwangi Brigjen T…",null,0,0,0,0,98,"""Thu Oct 08 12:54:48 +0000 2026""","""in""",0,false,null,2108179303100002598,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'allow_download_st…",null,"""{}""","""{}""",null,null,null,null,false,1,null
"""tweet""",2108178863297859718,"""https://x.com/kkpgoid/status/2…","""https://twitter.com/kkpgoid/st…","""Menteri Trenggono @saktitrengg…",null,0,0,0,0,111,"""Thu Oct 08 12:53:03 +0000 2026""","""in""",0,true,2108178427111211244,2108178427111211244,2493957026,"""kkpgoid""",false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'hashtags': [{'indices': [220…","""{'rest_id': '2493957026', 'res…",null,null,null,false,1,null
"""tweet""",2108178461231874433,"""https://x.com/BpsipSulut/statu…","""https://twitter.com/BpsipSulut…","""BRMP Sulut Dampingi Koperasi U…",null,0,0,0,0,13,"""Thu Oct 08 12:51:27 +0000 2026""","""in""",0,false,null,2108178461231874433,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'display_url': 'pi…",null,"""{}""","""{'urls': [{'display_url': 'fac…",null,null,null,null,false,1,null


In [50]:
df = df.select("text")
df

text
str
"""Pembalap MotoGP Fermin Aldegue…"
"""ngeliat banyaknya reaksi ""ajai…"
"""Terpantau hanya akun ini yang …"
"""Baca di sini: https://t.co/fED…"
"""Pahlawan Timnas Indonesia Juar…"
…
"""@RMOFootball @FIFAWorldCup Min…"
"""@idextratime TUNGGU KAMI ARAB …"
"""liat spain angkat trofi piala …"


In [51]:
df.dtypes

[String]

## Noise Removal

Noise Removal digunakan untuk membersihkan atau menghapus elemen yang biasanya tidak dibutuhkan dalam analisis teks, seperti HTML, URL, alamat email, mention, karakter kontrol, dan spasi berlebih yang tidak relevan dan tidak memiliki nilai informasi untuk analisis teks.

Contoh yang dibersihkan: Tanda baca, emoji, tag HTML, URL, karakter khusus (seperti @, #, $), dan spasi berlebih (whitespace).

### Code

In [52]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF\uFE0F\u200d]")

In [53]:
def noise_removal(df, text_col, output_col="text_clean"):
    def clean(text):
        if text is None: return ""
        text = html.unescape(str(text))
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+|\b\w+\.(?:co|com|id)/\S+", " ", text)
        text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)
        text = re.sub(r"(?<!\w)@\w+", " ", text)
        text = re.sub(r"^\s*RT\b", " ", text)
        # hashtag: buang '#', pisahkan CamelCase
        text = re.sub(r"#(\w+)", lambda m: " " + re.sub(r"(?<=[a-z])(?=[A-Z])", " ", m.group(1)) + " ", text)
        text = EMOJI.sub(" ", text)
        text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, clean)

ROMAN = re.compile(r"^m{0,3}(cm|cd|d?c{0,3})(xc|xl|l?x{0,3})(ix|iv|v?i{0,3})$")
def _collapse(m):
    w = m.group(0)
    if ROMAN.match(w) or w == "www":      # III, XXX, www jangan diubah
        return w
    return re.sub(r"(\w)\1{2,}", r"\1", w)

In [54]:
df_noise = noise_removal(
    df, text_col="text"
)

In [55]:
df_noise

text,text_clean
str,str
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…"
…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…"
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …"


In [56]:
print(type(df))


<class 'polars.dataframe.frame.DataFrame'>


## Normalisasi Teks

Normalisasi teks adalah proses menyeragamkan format teks, terutama huruf kapital, Unicode, dan spasi agar memiliki struktur linguistik yang seragam, terutama untuk teks tidak formal, bahasa gaul, singkatan, atau typo (typhographical errors).

Contoh: Mengubah kata "bgt" menjadi "banget", "udh" menjadi "sudah", atau menstandarisasi penulisan angka dan simbol mata uang.

### Code

In [61]:
SLANG_TIMNAS = {
    "yg": "yang",
    "dg": "dengan",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "karna": "karena",
    "tdk": "tidak",
    "gak": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "gk": "tidak",
    "enggak": "tidak",
    "ngga": "tidak",
    "bkn": "bukan",
    "bkan": "bukan",
    "sdh": "sudah",
    "udh": "sudah",
    "udah": "sudah",
    "blm": "belum",
    "jg": "juga",
    "jga": "juga",
    "sm": "sama",
    "sma": "sama",
    "klo": "kalau",
    "kalo": "kalau",
    "klu": "kalau",
    "trus": "terus",
    "tp": "tapi",
    "tpi": "tapi",
    "aja": "saja",
    "bgt": "banget",
    "emg": "memang",
    "emang": "memang",
    "org": "orang",
    "ornag": "orang",
    "dr": "dari",
    "dlm": "dalam",
    "pd": "pada",
    "sbg": "sebagai",
    "spt": "seperti",
    "mknya": "makanya",
    "stlah": "setelah",
    "ko": "kok",
    "thn": "tahun",
    "ehh": "eh",
    "wajarlah": "wajar lah",
    "gw": "saya",
    "gue": "saya",
    "gua": "saya",
    "lu": "kamu",
    "loe": "kamu",
    "indo": "indonesia",
    "king": "raja",
    "sty": "shin tae yong",
    "pssi": "persatuan sepakbola seluruh indonesia",
    "sugbk": "stadion utama gelora bung karno",
    "gbk": "gelora bung karno",
    "aff": "asean football federation",
    "rg": "indonesia",
    "moga": "semoga",
    "moga2": "semoga",
    "mantab": "mantap",
    "mantaap": "mantap",
    "ancur": "hancur",
    "liat": "lihat",
}

In [62]:
def text_normalization(df, text_col, output_col="text_normalized", slang=SLANG):
    def norm(text):
        if not text: return ""
        text = unicodedata.normalize("NFKC", text).lower()
        text = re.sub(r"[“”„‟]", '"', text)
        text = re.sub(r"[‘’‚‛]", "'", text)
        text = re.sub(r"[—–]", " - ", text)
        text = re.sub(r"\b\w*(\w)\1{2,}\w*\b", _collapse, text)       # mantaaap -> mantap (angka Romawi dikecualikan)
        text = re.sub(r"\brp\s?(\d)", r"rp \1", text)                # rp76 -> rp 76
        text = re.sub(r"\b([a-z]+)2\b", r"\1-\1", text)             # tiba2 -> tiba-tiba
        text = re.sub(r"\b[a-z0-9]+\b", lambda m: slang.get(m.group(0), m.group(0)), text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, norm)

In [63]:
df_normalized = text_normalization(
    df_noise, text_col="text_clean"
)

In [64]:
df_normalized

text,text_clean,text_normalized
str,str,str
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…"
…,…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…","""min, panggil madrid suruh spar…"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…","""tunggu kami arab dkk... king i…"
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …"


## Tokenisasi

Memecah kalimat atau paragraf yang panjang menjadi unit-unit kecil yang disebut token (biasanya berupa kata atau frasa), hasilnya berupa list, bukan string..

Contoh: Kalimat "Saya sedang belajar NLP." dipecah menjadi: ["Saya", "sedang", "belajar", "NLP"].

### Code

In [65]:
# def tokenization(df, text_col, output_col="tokens"):
#     """
#     Memecah teks menjadi token kata.
#     Output setiap baris berupa list.
#     """
#     result = df.clone()

#     def tokenize_text(text):
#         if text is None:
#             return ""
#         # if pl.isna(text):
#         #     return []

#         text = str(text)

#         # Ambil rangkaian karakter alfanumerik.
#         # Kata dengan apostrof atau tanda hubung dipisah.
#         return re.findall(r"\b\w+\b", text, flags=re.UNICODE)

#     return df.with_columns(
#         df[text_col].map_elements(tokenize_text).alias(output_col)
#     )
#     # result[output_col] = result[text_col].apply(tokenize_text)
#     # return result

In [66]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")
def tokenization(df, text_col, output_col="tokens"):
    return df.with_columns(
        pl.col(text_col).map_elements(lambda t: TOKEN_RE.findall(t or ""),
                                      return_dtype=pl.List(pl.Utf8)).alias(output_col))


In [67]:
df_tokens = tokenization(
    df_normalized, text_col="text_normalized"
)

In [68]:
df_tokens

text,text_clean,text_normalized,tokens
str,str,str,list[str]
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…","[""pahlawan"", ""timnas"", … ""bangkok""]"
…,…,…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…","""min, panggil madrid suruh spar…","[""min"", ""panggil"", … ""skrg""]"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…","""tunggu kami arab dkk... king i…","[""tunggu"", ""kami"", … ""otw""]"
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","[""liat"", ""spain"", … ""wok""]"


## Stopwords Removal

Menghapus kata-kata umum yang sering muncul dalam suatu bahasa namun memiliki sedikit atau bahkan tidak memiliki makna analitik yang kuat.

Contoh: dan, yang, di, dari, ke, untuk

### Code

In [75]:
# Stopword bahasa Indonesia
stop_words = set(stopwords.words("indonesian"))

# Tambahan stopword opsional
custom_stopwords = {
    "yg", "dg", "dgn", "nya", "sih", "dong",
    "nih", "deh", "aja", "udah", "gak", "nggak"
}

stop_words.update(custom_stopwords)

In [76]:
print(stop_words)

{'mengetahui', 'cukup', 'haruslah', 'melakukan', 'dong', 'semampu', 'melainkan', 'memang', 'diperbuatnya', 'semakin', 'enggaknya', 'diucapkannya', 'perlukah', 'sela', 'bulan', 'mirip', 'sebanyak', 'sinilah', 'ungkap', 'sajalah', 'memberi', 'caranya', 'kepada', 'tanpa', 'kalaulah', 'begitukah', 'menanti', 'diantaranya', 'seharusnya', 'bahkan', 'terdahulu', 'selamanya', 'dimulailah', 'diperbuat', 'malahan', 'seolah-olah', 'sekadarnya', 'hari', 'menuturkan', 'sangatlah', 'bagian', 'se', 'tiba-tiba', 'kurang', 'bolehkah', 'andalah', 'saya', 'setengah', 'dalam', 'usai', 'kemungkinannya', 'dahulu', 'diungkapkan', 'panjang', 'agak', 'terjadilah', 'menanti-nanti', 'sangat', 'benar', 'percuma', 'bermacam-macam', 'maupun', 'pastilah', 'bertanya-tanya', 'dijelaskan', 'memungkinkan', 'tadinya', 'betulkah', 'belakangan', 'dipastikan', 'kapanpun', 'mampu', 'artinya', 'sekiranya', 'disampaikan', 'berakhir', 'terlihat', 'cukupkah', 'jelaskan', 'diberi', 'seorang', 'inilah', 'tegasnya', 'kalian', 'ituk

In [77]:
def build_stopwords(keep_negation=True):
    sw = set(stopwords.words("indonesian"))
    sw |= {"sih", "dong", "nih", "deh", "kok", "lah", "kan", "nya", "tuh", "nah", "yah", "ya"}
    if keep_negation:
        sw -= {"tidak", "bukan", "tak", "belum", "jangan", "tanpa"}
    return sw

def stopword_removal(df, text_col, output_col="tokens_no_stopwords", stop_words=None):
    stop_words = stop_words or build_stopwords()
    return _apply_list(df, text_col, output_col,
                       lambda toks: [t for t in toks if t not in stop_words and len(t) > 1])

In [78]:
df_no_stopwords = stopword_removal(
    df_tokens, text_col="tokens"
)

In [74]:
df_no_stopwords

text,text_clean,text_normalized,tokens,tokens_no_stopwords
str,str,str,list[str],list[str]
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]"
…,…,…,…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…","""min, panggil madrid suruh spar…","[""min"", ""panggil"", … ""skrg""]","[""min"", ""panggil"", … ""skrg""]"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…","""tunggu kami arab dkk... king i…","[""tunggu"", ""kami"", … ""otw""]","[""tunggu"", ""arab"", … ""otw""]"
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","[""liat"", ""spain"", … ""wok""]","[""liat"", ""spain"", … ""wok""]"


In [79]:
df_no_stopwords['tokens_no_stopwords'][6]

""
str
"""komentar"""
"""shin"""
"""tae-yong"""
"""timnas"""
"""indonesia"""
…
"""cup"""
"""2026"""
"""jangan"""


## Stemming

Proses memotong imbuhan (prefiks, sufiks, infiks) pada sebuah kata untuk dikembalikan ke bentuk dasarnya (stem), tanpa selalu memperhatikan kaidah kamus yang valid.

Contoh: Kata "memakan", "dimakan", dan "termakan" akan diubah menjadi akar kata dasar "makan".

### Code

In [80]:
# Inisialisasi stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()


In [81]:
import polars as pl

def to_list(v):
    """Mengubah pl.Series atau iterables lainnya menjadi list Python."""
    if v is None:
        return []
    if isinstance(v, pl.Series):
        return v.to_list()
    if isinstance(v, list):
        return v
    return list(v)

def stemming(df: pl.DataFrame, text_col: str, output_col: str = "tokens_stemmed"):
    """
    Melakukan stemming menggunakan Sastrawi dengan aman dari noise metadata.
    """
    def stem_text(value):
        # Konversi nilai baris (bisa string, list, atau pl.Series) menjadi list token yang bersih
        if value is None:
            tokens = []
        elif isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            # Jika berupa string teks biasa (belum berupa list token)
            tokens = str(value).split()

        # Lakukan stemming pada setiap token
        return [stemmer.stem(str(token)) for token in tokens if token is not None]

    return df.with_columns(
        df[text_col].map_elements(
            stem_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [82]:
df_stemmed = stemming(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [83]:
df_stemmed

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed
str,str,str,list[str],list[str],list[str]
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]","[""balap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]","[""ngeliat"", ""banyak"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]","[""pantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]"
…,…,…,…,…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…","""min, panggil madrid suruh spar…","[""min"", ""panggil"", … ""skrg""]","[""min"", ""panggil"", … ""skrg""]","[""min"", ""panggil"", … ""skrg""]"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…","""tunggu kami arab dkk... king i…","[""tunggu"", ""kami"", … ""otw""]","[""tunggu"", ""arab"", … ""otw""]","[""tunggu"", ""arab"", … ""otw""]"
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","[""liat"", ""spain"", … ""wok""]","[""liat"", ""spain"", … ""wok""]","[""liat"", ""spain"", … ""wok""]"


In [ ]:
# STEM_EXCEPTIONS = {"belasan": "belas"}
# PROTECTED = {"wantimpres", "gibran", "prabowo", "rocky", "gerung", "pratikno", "dpr"}

In [ ]:
# _stemmer = StemmerFactory().create_stemmer()
# @lru_cache(maxsize=None)
# def _stem(tok):
#     if tok in PROTECTED or tok.isdigit() or "-" in tok: return tok
#     if tok in STEM_EXCEPTIONS: return STEM_EXCEPTIONS[tok]
#     return _stemmer.stem(tok)

# def stemming(df, text_col, output_col="tokens_stemmed"):
#     return _apply_list(df, text_col, output_col, lambda toks: [_stem(t) for t in toks])

In [84]:
df_stemmed['tokens_stemmed'][0]


""
str
"""balap"""
"""motogp"""
"""fermin"""
"""aldeguer"""
"""raya"""
…
"""utama"""
"""mandalika"""
"""musim"""


## Lemmatization

Proses mengubah kata berimbuhan menjadi bentuk dasarnya berdasarkan kamus dan tata bahasa yang valid (lemma), dengan memperhatikan konteks kalimat. Berbeda dengan stemming, proses ini idealnya menghasilkan bentuk lema yang valid secara linguistik.

Isi kamus lemma sebaiknya ditentukan dari kata berimbuhan yang benar-benar sering muncul dari data, bukan contoh umum. aturan untuk menentukan niai lema bisasnya menggunakan beberaoa ruls seperti:
1. Kalau imbuhan hanya mengubah bentuk gramatikal tanpa mengubah makna inti (dilantik → lantik), pakai kata dasar.

2. Kalau kata berimbuhan sudah punya makna sendiri yang berbeda dari kata dasarnya (pemerintah, kebijakan, anggaran, alasan), pertahankan bentuk aslinya sebagai lemma.

### Cek

### Code

In [91]:
lemma_dictionary = {
    "menjuarai": "juara",
    "dijuarai": "juara",
    "kejuaraan": "juara",
    "menang": "menang",
    "kemenangan": "menang",
    "memenangkan": "menang",
    "dimenangkan": "menang",
    "bermain": "main",
    "memainkan": "main",
    "dimainkan": "main",
    "permainan": "main",
    "pemain": "main",
    "pertandingan": "tanding",
    "bertanding": "tanding",
    "menandingi": "tanding",
    "melawan": "lawan",
    "perlawanan": "lawan",
    "berlawanan": "lawan",
    "membawa": "bawa",
    "dibawa": "bawa",
    "bawaan": "bawa",
    "meraih": "raih",
    "diraih": "raih",
    "mencetak": "cetak",
    "dicetak": "cetak",
    "mengalahkan": "kalah",
    "dikalahkan": "kalah",
    "kekalahan": "kalah",
    "mendukung": "dukung",
    "didukung": "dukung",
    "dukungan": "dukung",
    "pendukung": "dukung",
    "membanggakan": "bangga",
    "kebanggaan": "bangga",
    "melatih": "latih",
    "dilatih": "latih",
    "pelatih": "latih",
    "pelatihan": "latih",
    "berjuang": "juang",
    "perjuangan": "juang",
    "pejuang": "juang",
    "merayakan": "rayak",
    "perayaan": "rayak",
    "dirayakan": "rayak",
    "menonton": "nonton",
    "penonton": "nonton",
    "ditonton": "nonton",
    "menerima": "terima",
    "diterima": "terima",
    "terima": "terima",
    "penerima": "terima",
}


In [92]:

def lemmatization(
    df: pl.DataFrame,
    text_col: str,
    output_col: str = "tokens_lemmatized",
    lemma_dict: dict = None
):
    """
    Lemmatization berbasis kamus.
    Kata yang tidak ada dalam kamus dipertahankan.
    """
    if lemma_dict is None:
        lemma_dict = lemma_dictionary

    def lemmatize_text(value):
        if value is None:
            return []
        # kolom list di Polars masuk sebagai pl.Series -> ubah ke list Python
        if isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            tokens = str(value).split()

        return [lemma_dict.get(str(t).lower(), str(t).lower()) for t in tokens]

    return df.with_columns(
        pl.col(text_col).map_elements(
            lemmatize_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [93]:
df_lemmatized = lemmatization(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [94]:
df_lemmatized

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_lemmatized
str,str,str,list[str],list[str],list[str]
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]","[""ngeliat"", ""banyaknya"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]","[""terpantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]"
…,…,…,…,…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…","""min, panggil madrid suruh spar…","[""min"", ""panggil"", … ""skrg""]","[""min"", ""panggil"", … ""skrg""]","[""min"", ""panggil"", … ""skrg""]"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…","""tunggu kami arab dkk... king i…","[""tunggu"", ""kami"", … ""otw""]","[""tunggu"", ""arab"", … ""otw""]","[""tunggu"", ""arab"", … ""otw""]"
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","[""liat"", ""spain"", … ""wok""]","[""liat"", ""spain"", … ""wok""]","[""liat"", ""spain"", … ""wok""]"


## Hasil

In [95]:
columns_to_view = [
    "text",
    "text_clean",
    "text_normalized",
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

df_comparison = (
    df_lemmatized
    .with_columns(df_stemmed["tokens_stemmed"])
    .select(columns_to_view)
)

df_comparison.head()

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,list[str],list[str],list[str],list[str]
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","[""pembalap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]","[""balap"", ""motogp"", … ""sabrina""]","[""pembalap"", ""motogp"", … ""sabrina""]"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","[""ngeliat"", ""banyaknya"", … ""belakang""]","[""ngeliat"", ""banyaknya"", … ""negara""]","[""ngeliat"", ""banyak"", … ""negara""]","[""ngeliat"", ""banyaknya"", … ""negara""]"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","[""terpantau"", ""hanya"", … ""si""]","[""terpantau"", ""akun"", … ""si""]","[""pantau"", ""akun"", … ""si""]","[""terpantau"", ""akun"", … ""si""]"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","[""baca"", ""di"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]","[""baca"", ""calvin"", … ""2026100815292938""]"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]","[""pahlawan"", ""timnas"", … ""bangkok""]"


In [96]:
df_export = df_comparison.clone()

token_columns = [
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

# Ubah list token menjadi string
for col in token_columns:
    df_export = df_export.with_columns(
        df_export[col].map_elements(
            lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens),
            return_dtype=pl.Utf8
        ).alias(col)
    )

# # Simpan hasil ke CSV
# df_export.write_csv(
#     "hasil_preprocessing.csv",
#     encoding="utf8"   # Polars tidak pakai "utf-8-sig", cukup "utf8"
# )


In [97]:
df_export = df_comparison.with_columns(
    [pl.col(c).list.join(" ") for c in token_columns]
)

df_export.write_csv("hasil_preprocessingtimnas.csv")

In [98]:
df_export.write_csv("hasil_preprocessingtimnas.csv")


In [99]:
df_load = pl.read_csv('hasil_preprocessingtimnas.csv')

In [100]:
df_load

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,str,str,str,str
"""Pembalap MotoGP Fermin Aldegue…","""Pembalap MotoGP Fermin Aldegue…","""pembalap motogp fermin aldegue…","""pembalap motogp fermin aldegue…","""pembalap motogp fermin aldegue…","""balap motogp fermin aldeguer r…","""pembalap motogp fermin aldegue…"
"""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ""ajai…","""ngeliat banyaknya reaksi ajaib…","""ngeliat banyaknya reaksi ajaib…","""ngeliat banyak reaksi ajaib or…","""ngeliat banyaknya reaksi ajaib…"
"""Terpantau hanya akun ini yang …","""Terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","""terpantau hanya akun ini yang …","""terpantau akun tidak reaksi ti…","""pantau akun tidak reaksi timna…","""terpantau akun tidak reaksi ti…"
"""Baca di sini: https://t.co/fED…","""Baca di sini: Calvin Verdonk t…","""baca di sini: calvin verdonk t…","""baca di sini calvin verdonk ta…","""baca calvin verdonk tak meraya…","""baca calvin verdonk tak raya h…","""baca calvin verdonk tak rayak …"
"""Pahlawan Timnas Indonesia Juar…","""Pahlawan Timnas Indonesia Juar…","""pahlawan timnas indonesia juar…","""pahlawan timnas indonesia juar…","""pahlawan timnas indonesia juar…","""pahlawan timnas indonesia juar…","""pahlawan timnas indonesia juar…"
…,…,…,…,…,…,…
"""@RMOFootball @FIFAWorldCup Min…","""Min, panggil madrid suruh spar…","""min, panggil madrid suruh spar…","""min panggil madrid suruh spari…","""min panggil madrid suruh spari…","""min panggil madrid suruh spari…","""min panggil madrid suruh spari…"
"""@idextratime TUNGGU KAMI ARAB …","""TUNGGU KAMI ARAB DKK... KING I…","""tunggu kami arab dkk... king i…","""tunggu kami arab dkk king indo…","""tunggu arab dkk king indo otw""","""tunggu arab dkk king indo otw""","""tunggu arab dkk king indo otw"""
"""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …","""liat spain angkat trofi piala …"
